[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/15_sft_checkpoint.ipynb)

# 15 · 共享 SFT 检查点 S1（只训练一次，11 / 12 / 13 都从它开始）

**先跑这个 notebook。** 它训练一个 SFT 检查点，合并后推到你的 HF 私有仓库 `<你的用户名>/qwen3-4b-s1-sft`；
notebook 11（GRPO）、12（DPO）、13（蒸馏）的 `INIT_MODEL="auto"` 会自动从这个仓库加载，不需要再做 SFT。

## 为什么换数据配方（依据见 `experiments/DATASETS.md`）

X1b 的 SFT 让模型学会了收尾，但回答是"背一段知识 + 答案"，有答案的题里的准确率几乎没变（29% → 31%），GRPO 也没有推理可以强化。原因在数据：
2 万条里科学只有 8500 条，选择题样本是 SciQ 的"段落 + 答案"。这一版按论文里的证据重新配：

| 证据 | 结论 | 这里怎么做 |
|---|---|---|
| MegaScience（arXiv 2507.16812，Qwen2.5-7B）：TextbookReasoning 单独训练 GPQA-D 34.3，是公开科学数据里最高的；整个混合 36.4；直接蒸馏 R1 长思维链的数据只有 29–30 | 大学教材题 + **核对过的短分步解答** 比长思维链更有效，也更省 token | 教材题（物理 / 化学 / 生物为主）占一半 |
| 同一论文：选择题数据（Nemotron-Science）让选择题基准变好、计算题变差；按难度筛选最好；它的 LLM 去污染查出 16.4 万条和基准重复的题 | 选择题要有，但不能只有选择题；用去污染后的版本 | 只用 MegaScience 里筛过的 4 选 1 题，改写成 GPQA 的题面，结尾 `The answer is (X).` |
| OpenThoughts3（arXiv 2506.04178）：领域内数据最重要，混入科学数据后数学 / 代码的迁移收益消失；每个领域混两个最好的来源就够 | 以科学为主，数学只留一点 | 科学约 80%，数学约 15% |
| Dr.SCI（arXiv 2602.08321，**同一个 Qwen3-4B-Base**）：5 万条非思考数据 ≈ 1 万条思考数据（GPQA-D 43.6 vs 42.3）；按 4-gram 覆盖度选数据 +2 分 | 同样的 token 预算下，短解答不吃亏 | 每个部分先抽 2 倍候选，再按覆盖度挑 |
| Dolci-Think 抽样：科学思维链中位数约 7200 token，数学约 2 万 | 长思维链很贵（SFT、评测、GRPO 都按 token 算钱） | S1 不用；`VARIANT="think"` 单独做消融 |

## 数据配方（`a100`，约 10 万条）

| 部分 | 来源 | 条数 | 作用 |
|---|---|---|---|
| `mcq` | MegaScience 的 4 选 1 科学题（题面改成 GPQA 格式；只留"两份解答答案一致"的题；选项字母配平） | 约 8000 | 教模型在 GPQA 题面下逐个分析选项，再写答案 |
| `textbook_*` | MegaScience / TextbookReasoning：物理 2 万、化学 2 万、生物 1.5 万、医学 5000、数学 8000 | 6.8 万 | 大学水平的知识 + 短分步解答 |
| `dolci_science` | Dolci-Instruct 的 OpenThoughts3 科学题（StackExchange 物理、有机化学）+ SciRIFF | 1.6 万 | 研究生水平的物理 / 有机化学 |
| `natural` | MegaScience / NaturalReasoning | 8000 | 通用推理，解答较长 |
| `dolci_math` | Dolci-Instruct 的 Tulu 3 数学 | 8000 | 计算 |

4 选 1 题按题目哈希分成三份：55% 给这里的 SFT，35% 留给 GRPO / DPO 当题库，10% 当验证集——后两份的解答模型没见过。

## 生成预算和思维链消融（`VARIANT="think"`）

排行榜的锁定评测保持 **8192**。参考分支可以放宽到 **16384**（Qwen3-4B 的上下文上限是 32768；按显存估算，T4 上 4B 最多放得下 2 万出头的上下文，所以 16384 基本也是 T4 的上限）。
S1 的回答只有几百到一千多 token，放宽对它没有意义，所以 S1 仍按 8192 评测。放宽是为了回答另一个问题：**长思考能加多少分？**

`VARIANT="think"`（S1 跑完之后再跑，约 1.5 小时训练 + 1 小时评测）：从 S1 接着训练 5000 条 Dolci-Think 科学思维链（OpenThoughts3 科学题，QwQ-32B 的 `<think>` 推理，
取 ≤8192 token 的；全集中位数约 7200）+ 2000 条 GPQA 题面的选择题（把分步解答放进 `<think>…</think>`，后面仍是 `The answer is (X).`）。
训练前后都在 Diamond 上按 16384 评测，同时报告 `acc_flex_at_8192`（同样的回答在排行榜预算下能得多少分）和 `over_8192_rate`（多少回答超过 8192）。
为什么不把 S1 直接换成思维链：1 万条思维链就要约 3 小时 SFT；学会长思考后每次评测从 10 分钟变成 1–3 小时，GRPO 每个回答贵约 8 倍——剩下的额度做不完下游。

## 时间和算力

`a100`：数据准备约 20–30 分钟，训练约 3–4 小时（按 X1b 实测的 6.2k token/秒估），评测约 15 分钟。
- **不用守着**：Run all 之后可以离开。检查点、adapter、阶段结果都存在 Google Drive（开头会弹一次授权），断线后重新 Run all 会跳过已完成的阶段、从最近的检查点续训；
  全部跑完自动断开 runtime（`AUTO_DISCONNECT`）。
- **改了配置想重跑某一格**：直接重跑，不用重启。加载模型那一格会复用已经加载的模型；只有换模型 / 上下文长度 / LoRA rank 才需要 Runtime → Restart session（会有明确提示）。
- **时间上限**：`MAX_TRAIN_HOURS` 到了就停止训练，照常合并、推送、评测。
- 先用 `MODE="quick"`（0.6B，约 15 分钟，T4 也行）跑一遍确认流程，再上 A100。

⚠️ 快速评测不是排行榜分数。A100 上训练的模型提交时注明 A100（不参与排名）。

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0


In [2]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码都在 scripts/ 里）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
if os.path.basename(os.getcwd()) != "GPQA-eval":   # 重跑这一格时不要再进一层
    %cd GPQA-eval
!git pull -q
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import recipe_utils
print("reloaded:", recipe_utils.reload_scripts())   # 改了 scripts/ 之后重跑这一格即可，不用重启 runtime

/content/GPQA-eval
reloaded: ['recipe_utils']


In [3]:
# 安装（约 3–5 分钟）。2026-11 实测可用：unsloth 2026.11.3、vllm 0.31、trl 1.15、transformers 5.17（A100）
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

Using Python 3.13.16 environment at: /usr
Resolved 227 packages in 1.01s
Prepared 91 packages in 17.20s
Uninstalled 13 packages in 980ms
Installed 91 packages in 289ms
 + agent-detector==2.0.0
 + anthropic==2.8.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + bitsandbytes==0.50.2
 + blake3==1.0.11
 + cbor2==6.2.1
 + compressed-tensors==0.17.0
 - cuda-core==0.7.0
 + cuda-core==1.3.0
 + cuda-tile==1.8.1
 + cut-cross-entropy==25.1.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.9.2
 + email-validator==2.3.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.33
 + fastapi-cloud-cli==0.28.0
 + fastar==0.12.0
 + fastsafetensors==0.5.0
 + flashinfer-python==0.7.0.post1
 + hf-transfer==0.1.9
 + humming-kernels==0.1.16
 + ijson==3.6.0
 + instanttensor==0.2.3
 + interegular==0.3.3
 + jmespath==1.1.0
 + json5==0.17.3
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.46.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.8.0
 + mcp==2.3.0
 + mcp-ty

In [4]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if torch.cuda.get_device_capability()[0] < 8:   # T4 不支持 FlashInfer 采样器
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

NVIDIA A100-SXM4-80GB 79.3 GB
{'unsloth': '2026.11.4', 'vllm': '0.31.0', 'trl': '1.15.0', 'transformers': '5.17.0', 'torch': '2.13.0', 'peft': '0.21.2', 'wandb': '0.28.1'}


In [5]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需，要有 **write** 权限：最后要把模型推到你的私有仓库；
#                      还要先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（建议；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
from recipe_utils import sft_repo_name
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF user: darcyyao | W&B key found


## 0. 配置

In [6]:
VARIANT      = "s1"                           # "s1" = 共享 SFT 检查点（先跑）| "think" = 思维链消融：从 S1 接着训练长思维链（S1 跑完之后再跑）
EXP_NAME     = None                           # None = "S1" / "S1think"
MODE         = "a100"                         # "quick"（0.6B，验证流程）| "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"                           # None = 用预设（a100: unsloth/Qwen3-4B-Base）
CHAT_STYLE   = "olmo"                         # 回答以 <|endoftext|> 结束（X1b 已验证：截断率 97% → 3–5%）
SFT_REPO     = None                           # None = f"{HF_USER}/qwen3-4b-s1-sft"（私有；think 变体是 ...-s1think-sft）；11 / 12 / 13 的 INIT_MODEL="auto" 找的是前者
GEN_BUDGET   = None                           # 评测的生成预算。None = s1: 8192（和排行榜一致）；think: 16384（只在参考分支放宽）
MAX_TRAIN_HOURS = 5.0                         # 本次会话的训练时间上限；到了就停，照常合并 / 推送 / 评测
USE_DRIVE    = True                           # 检查点、adapter、阶段结果都存到 Google Drive：runtime 重置后不会丢
RESUME       = True                           # 已经完成的阶段（数据、训练、推送、评测）直接读结果，不重跑；False = 全部重来
AUTO_DISCONNECT = True                        # 全部跑完后自动断开 runtime（不再消耗 CU）。想留着 runtime 调试就设 False
RUN_EVAL     = True                           # 训练后做 GPQA 快速评测（约 15 分钟）

# ---- 数据配方（a100 的条数；quick 自动按 scale 缩小）。依据见本 notebook 顶部和 experiments/DATASETS.md
MCQ_N       = 9000                            # MegaScience 4 选 1 科学题 → GPQA 题面（可用的大约 8000，全用）
TEXTBOOK    = dict(physics = 20000, chemistry = 20000, biology = 15000, medicine = 5000, math = 8000)
NATURAL_N   = 8000
DOLCI_DATASET = "allenai/Dolci-Instruct-SFT-7B"
DOLCI_EXCLUDE = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic", "tool"]
DOLCI_PARTS = [
    dict(part = "dolci_science", n = 16000, include = ["science", "sciriff"]),
    dict(part = "dolci_math",    n = 8000,  include = ["math", "gsm", "algebra"]),
]
# ---- VARIANT="think" 用的数据
THINK_N       = 5000                          # Dolci-Think 科学思维链（题目 + 回答 ≤ 8192 token）
THINK_MCQ_N   = 2000                          # GPQA 题面的选择题，解答放进 <think>…</think>
EXTRA_SFT     = []                            # 其它数据集，写法见 experiments/DATASETS.md
SFT_SELECT    = "coverage"                    # "coverage"（Dr.SCI 的 EESFT：每个部分从 POOL_FACTOR 倍候选里按 4-gram 覆盖度挑）| "random"
POOL_FACTOR   = 2
COVERAGE_NORM = "sqrt"                        # "none" = 完全照论文（偏向长样本）；"sqrt" = 除以 √长度

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(model="unsloth/Qwen3-0.6B-Base", base_model="Qwen/Qwen3-0.6B-Base",
                 scale=0.01, max_seq_len=2048, gpu_util=0.6, lora_rank=16, sft_max_tokens=1024,
                 lr=2e-4, batch=4, accum=2, max_steps=60, save_steps=30, warmup_steps=5,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, shuffle_buffer=1000,
                 think=dict(max_seq_len=4096, sft_max_tokens=2048, lr=1e-4, batch=2, accum=4, max_steps=30, eval_max_tokens=2048)),
    a100 = dict(model="unsloth/Qwen3-4B-Base", base_model="Qwen/Qwen3-4B-Base",
                scale=1.0, max_seq_len=12288, gpu_util=0.8, lora_rank=128, sft_max_tokens=2560,
                lr=1e-4, batch=8, accum=4, max_steps=-1, save_steps=400, warmup_steps=100,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=4,
                shuffle_buffer=20000,
                # 思维链消融：样本最长 8192 token → 每个 micro-batch 只放 2 条；评测放宽到 16384，只测 Diamond
                think=dict(max_seq_len=20480, lora_rank=64, sft_max_tokens=8192, lr=5e-5, batch=2, accum=16, save_steps=100,
                           warmup_steps=20, eval_configs=["gpqa_diamond"], eval_max_tokens=16384)),
)
assert VARIANT in ("s1", "think")
THINK = VARIANT == "think"
Q = dict(PRESETS[MODE])
_think = Q.pop("think")
if THINK:
    Q.update(_think)
if GEN_BUDGET:
    Q.update(eval_max_tokens = GEN_BUDGET, max_seq_len = max(Q["max_seq_len"], GEN_BUDGET + 4096))
EXP_NAME = EXP_NAME or ("S1think" if THINK else "S1")
MODEL_NAME = MODEL_NAME or Q["model"]
BASE_MODEL = Q["base_model"]
S1_REPO = sft_repo_name(HF_USER, MODEL_NAME)
INIT_MODEL = S1_REPO if THINK else None       # think 变体从 S1 接着训练
SFT_REPO = SFT_REPO or (S1_REPO.replace("-s1-sft", "-s1think-sft") if THINK else S1_REPO)
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK, SCALE = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"], Q["scale"]
print(EXP_NAME, VARIANT, MODE, "start from:", INIT_MODEL or MODEL_NAME, "->", SFT_REPO, Q)

S1 s1 a100 start from: unsloth/Qwen3-4B-Base -> darcyyao/qwen3-4b-s1-sft {'model': 'unsloth/Qwen3-4B-Base', 'base_model': 'Qwen/Qwen3-4B-Base', 'scale': 1.0, 'max_seq_len': 12288, 'gpu_util': 0.8, 'lora_rank': 128, 'sft_max_tokens': 2560, 'lr': 0.0001, 'batch': 8, 'accum': 4, 'max_steps': -1, 'save_steps': 400, 'warmup_steps': 100, 'eval_configs': ['gpqa_diamond', 'gpqa_main'], 'eval_n': None, 'eval_max_tokens': 8192, 'eval_samples': 4, 'shuffle_buffer': 20000}


In [7]:
# 持久目录：Google Drive（runtime 重置后还在）或本地。检查点、adapter、准备好的数据、各阶段结果都在这里
from recipe_utils import persist_dir, StageCache
CKPT_DIR = persist_dir(f"{EXP_NAME}_{MODE}", USE_DRIVE)
CACHE = StageCache(CKPT_DIR, enabled = RESUME)
RESUME = CACHE.check_config(dict(variant = VARIANT, model = MODEL_NAME, init_model = INIT_MODEL, chat_style = CHAT_STYLE, mcq_n = MCQ_N,
                                 textbook = TEXTBOOK, natural_n = NATURAL_N, dolci_parts = DOLCI_PARTS, extra_sft = EXTRA_SFT,
                                 think_n = THINK_N, think_mcq_n = THINK_MCQ_N, sft_select = SFT_SELECT, pool_factor = POOL_FACTOR,
                                 coverage_norm = COVERAGE_NORM,
                                 **{k: Q[k] for k in ("scale", "max_seq_len", "lora_rank", "sft_max_tokens", "lr", "batch", "accum",
                                                      "max_steps", "eval_n", "eval_max_tokens")}))
print("persist dir:", CKPT_DIR, "| finished stages:", CACHE.done(),
      "| checkpoints:", sorted(d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")))

[!] Google Drive 没挂上（ValueError）：结果只存在本地 outputs/S1_a100，runtime 重置后会丢。
persist dir: outputs/S1_a100 | finished stages: [] | checkpoints: []


## 1. 加载模型（`fast_inference=True`：训练后的评测用同一个 vLLM 引擎）

In [8]:
import os, json, random, statistics, time, torch
from collections import Counter
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
from recipe_utils import (RESPONSE_PARTS, STOP_STRS, TABLE_COLS, QuickEval, setup_chat, chat_prompt, log_eval, print_table,
                          decontam_rows, prepare_sft_sources, run_script, wandb_init, load_guard, restore_lora, end_runtime)
from select_coverage import select_per_part

# 这一格可以反复运行：模型已经加载过就直接复用（vLLM 引擎不能在同一个进程里建第二次，以前重跑会报错）。
_KEY = (INIT_MODEL or MODEL_NAME, MAX_SEQ_LEN, LORA_RANK, GPU_UTIL)
if load_guard(globals(), _KEY):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = INIT_MODEL or MODEL_NAME,
        max_seq_length = MAX_SEQ_LEN,
        load_in_4bit = False,
        fast_inference = True,
        max_lora_rank = LORA_RANK,
        gpu_memory_utilization = GPU_UTIL,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_RANK,                     # S1：一次性的 SFT 要学很多知识，rank 用 128（X1b 是 64）
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = LORA_RANK * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    _LOADED_KEY = _KEY
else:
    print("[i] 模型已经加载，直接复用（没有重新加载）。训练格开始时会把 LoRA 权重重置。")
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)
EOS_ID = tokenizer.eos_token_id
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
assert _demo.endswith(tokenizer.eos_token), "最后一个回合应以 eos 结束"
print(repr(_demo), "| eos:", tokenizer.eos_token, EOS_ID, "| stop ids:", STOP_IDS)

RUN_NAME = f"{EXP_NAME}-sft-{MODEL_NAME.split('/')[-1]}-{MODE}"



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 22:31:25.013000 1980 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 22:31:25.122000 1980 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 5.17.0. vLLM: 0.31.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.87875.
Unsloth: Patched vLLM's piecewise graph-partition pass so compilation_config=3 (CUDA graphs) works with LoRA.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_inductor` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `swap_space` is not supported in vLLM. Skipping.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
Unsloth: vLLM loading /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Base/snapshots/0573b584bc6b32adc84bb9c91bf9b71bea71fc40 with actual GPU utilization = 87.28%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.25 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 8192. Num Sequences = 128.
Unsloth: vLLM's KV Cache c

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


... (10 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm_2', 'cross_attn_input_layernorm', 'layer_norm1', 'cross_attn_post_attention_layernorm', 'post_feedforward_layernorm_2', 'norm', 'q_norm', 'attention_norm', 'post_per_layer_input_norm', 'pre_feedforward_layernorm', 'input_layernorm', 'ffn_norm', 'operator_norm', 'k_layernorm', 'norm1', 'post_feedforward_layernorm_1', 'post_attention_layernorm', 'layer_norm2', 'norm2', 'post_feedforward_layernorm', 'post_layernorm', 'k_norm', 'q_layernorm']


Unsloth 2026.11.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
'<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\nHello<|endoftext|>' | eos: <|endoftext|> 151643 | stop ids: [151643, 151645, 151644, 70564]


In [9]:
# 创建一个排除了 'model' 和 'base_model' 的 Q 字典备份，避免在 dict() 中重复传参
wandb_config_q = {k: v for k, v in Q.items() if k not in ["model", "base_model"]}

wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, dict(
    exp = EXP_NAME, recipe = "sft_checkpoint", variant = VARIANT, init_model = INIT_MODEL, think_mcq_n = THINK_MCQ_N,
    mode = MODE, model = MODEL_NAME, base_model = BASE_MODEL, sft_repo = SFT_REPO,
    chat_style = CHAT_STYLE, mcq_n = MCQ_N, textbook = TEXTBOOK, natural_n = NATURAL_N, dolci_parts = DOLCI_PARTS,
    think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT, pool_factor = POOL_FACTOR, coverage_norm = COVERAGE_NORM,
    max_train_hours = MAX_TRAIN_HOURS, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, **wandb_config_q), WANDB_ENTITY)

from vllm import SamplingParams
EVAL, RESULTS = None, {}
def quick_eval(tag, lora_request = None):
    r = EVAL.run(tag, lambda texts, kw: model.fast_generate(texts, sampling_params = SamplingParams(**kw), lora_request = lora_request),
                 cache = CACHE)
    log_eval(wandb, r, tag)
    for cfg, v in r.items():
        if v.get("extract_rate"):
            print(tag, cfg, "有答案的题里的准确率:", round(100 * v["acc_flex"] / v["extract_rate"], 1), "| 平均 token:", v["mean_tokens"])
if RUN_EVAL:
    EVAL = QuickEval.from_gpqa(tokenizer, Q["eval_configs"], Q["eval_n"], max_tokens = Q["eval_max_tokens"],
                               max_model_len = MAX_SEQ_LEN, samples = Q["eval_samples"], stop_ids = STOP_IDS)
    RESULTS = EVAL.results
    if THINK:
        quick_eval("0_s1")    # 起点（S1）在同样的预算下的分数；S1 的回答短，几分钟

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: darcy-yi-yao (darcy-yi-yao-apple) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B: https://wandb.ai/darcy-yi-yao-apple/gpqa-hackathon/runs/qhxa4m3o


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

gpqa_diamond.csv:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

gpqa_main.csv:   0%|          | 0.00/3.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/448 [00:00<?, ? examples/s]

## 2. 数据

1. `scripts/science_mix.py`：下载 MegaScience（约 1.9 GB），4 选 1 题改成 GPQA 题面并按题目哈希拆分，教材题按学科抽样；写出候选池和留出的题库。
2. `scripts/prepare_dolci_sft.py`：Dolci-Instruct 的科学 / 数学部分（流式读取）。
3. 全部过 GPQA 13-gram 去污染。
4. 每个部分从候选里按 4-gram 覆盖度挑到目标条数（`SFT_SELECT="coverage"`）。

准备好的训练集会存到检查点目录；续训时直接读它，保证和断线前是同一份数据、同一个顺序。

In [10]:
import json
import recipe_utils

# 动态修补 prepare_sft_sources 以在遇到损坏的 JSON 行时跳过，防止报错中断
_orig_prepare_sft_sources = recipe_utils.prepare_sft_sources
def patched_prepare_sft_sources(*args, **kwargs):
    try:
        return _orig_prepare_sft_sources(*args, **kwargs)
    except json.JSONDecodeError:
        # 如果是内部解析临时文件时崩了，我们提供一个安全的降级读取机制
        print("[!] 捕获到 JSONDecodeError，正在启用安全容错模式重新读取...")
        # 临时拦截标准 json.loads 使得它能容错跳过不完整的行
        orig_loads = json.loads
        def tolerant_loads(s, *a, **kw):
            try:
                return orig_loads(s, *a, **kw)
            except Exception:
                return {}
        json.loads = tolerant_loads
        try:
            res = _orig_prepare_sft_sources(*args, **kwargs)
            # 过滤掉由于解析失败返回的空行
            res = [r for r in res if r.get("messages")]
            return res
        finally:
            json.loads = orig_loads

recipe_utils.prepare_sft_sources = patched_prepare_sft_sources

DATA_FILE = f"{CKPT_DIR}/sft_train.jsonl"
DATA = {}
if RESUME and os.path.exists(DATA_FILE) and os.path.exists(f"{CKPT_DIR}/sft_data.json"):
    sft_rows = [json.loads(l) for l in open(DATA_FILE)]
    DATA = json.load(open(f"{CKPT_DIR}/sft_data.json"))
    print(f"[i] 读取已准备好的训练集：{len(sft_rows)} 条（{DATA_FILE}）")
elif THINK:
    # 思维链消融：Dolci-Think 科学思维链 + GPQA 题面的选择题（解答放进 <think>…</think>）
    from science_mix import think_wrap
    def _n(x):
        return max(1, int(x * SCALE))
    t0 = time.time()
    pool = prepare_sft_sources([dict(dataset = "allenai/Dolci-Think-SFT-7B", n = THINK_N, include = ["science"], part = "think",
                                     exclude = DOLCI_EXCLUDE)],
                               MODEL_NAME, Q["sft_max_tokens"], scale = SCALE, shuffle_buffer = min(Q["shuffle_buffer"], 5000))
    if THINK_MCQ_N > 0:
        run_script(["scripts/science_mix.py", "--out-dir", "data/s1_think", "--tokenizer", MODEL_NAME, "--max-tokens", 2560,
                    "--mcq", _n(THINK_MCQ_N), "--textbook", "--natural", 0, "--pool-factor", 1], show_tail = 5)
        for l in open("data/s1_think/sft_pool.jsonl"):
            r = json.loads(l)
            r["messages"][-1]["content"] = think_wrap(r["messages"][-1]["content"])
            pool.append(dict(r, part = "mcq_think"))
    pool, dropped = decontam_rows(pool, lambda r: {"messages": r["messages"]}, "s1_think_pool")
    sft_rows = pool
    random.Random(0).shuffle(sft_rows)
    _lens = sorted(r.get("n_tokens") or 0 for r in sft_rows if r.get("part") == "think")
    DATA = dict(selected = len(sft_rows), dropped_decontam = dropped, by_part = dict(Counter(r.get("part") for r in sft_rows)),
                tokens = sum(r.get("n_tokens") or 0 for r in sft_rows),
                think_tokens = dict(p50 = _lens[len(_lens) // 2], p90 = _lens[int(0.9 * len(_lens))], max = _lens[-1]) if _lens else {},
                sources = Counter(r.get("source") for r in sft_rows).most_common(10),
                prep_minutes = round((time.time() - t0) / 60, 1))
    with open(DATA_FILE, "w") as f:
        for r in sft_rows:
            f.write(json.dumps({"messages": r["messages"], "part": r.get("part"), "n_tokens": r.get("n_tokens")}, ensure_ascii = False) + "\n")
    json.dump(DATA, open(f"{CKPT_DIR}/sft_data.json", "w"), indent = 1, default = str)
else:
    def _n(x):
        return max(1, int(x * SCALE))
    os.makedirs("data/s1", exist_ok = True)
    t0 = time.time()
    run_script(["scripts/science_mix.py", "--out-dir", "data/s1", "--tokenizer", MODEL_NAME, "--max-tokens", Q["sft_max_tokens"],
                "--mcq", _n(MCQ_N), "--textbook", *[f"{k}={_n(v)}" for k, v in TEXTBOOK.items()], "--natural", _n(NATURAL_N),
                "--pool-factor", POOL_FACTOR if SFT_SELECT == "coverage" else 1], show_tail = 60)
    MIX_STATS = json.load(open("data/s1/mix_stats.json"))
    pool = [json.loads(l) for l in open("data/s1/sft_pool.jsonl")]
    targets = {"mcq": _n(MCQ_N), "natural": _n(NATURAL_N), **{f"textbook_{k}": _n(v) for k, v in TEXTBOOK.items()}}

    pf = POOL_FACTOR if SFT_SELECT == "coverage" else 1
    dolci_specs = [dict(dataset = DOLCI_DATASET, n = d["n"], include = d["include"], exclude = DOLCI_EXCLUDE, part = d["part"])
                   for d in DOLCI_PARTS]
    dolci_specs += [dict(s, part = s.get("part", "extra")) for s in EXTRA_SFT]
    pool += prepare_sft_sources(dolci_specs, MODEL_NAME, Q["sft_max_tokens"], scale = SCALE * pf, shuffle_buffer = Q["shuffle_buffer"])
    for s in dolci_specs:
        targets[s["part"]] = targets.get(s["part"], 0) + _n(s["n"])

    pool, dropped = decontam_rows(pool, lambda r: {"messages": r["messages"]}, "s1_pool")
    sft_rows, SELECT = select_per_part(pool, targets, method = SFT_SELECT, norm = COVERAGE_NORM, seed = 0)
    random.Random(0).shuffle(sft_rows)
    DATA = dict(pool = len(pool), selected = len(sft_rows), dropped_decontam = dropped, select = SFT_SELECT, by_part = SELECT,
                tokens = sum(r.get("n_tokens") or 0 for r in sft_rows), mcq = MIX_STATS.get("mcq"),
                textbook_subjects = MIX_STATS.get("textbook_subjects"),
                sources = Counter(r.get("source") for r in sft_rows).most_common(15),
                prep_minutes = round((time.time() - t0) / 60, 1))
    with open(DATA_FILE, "w") as f:
        for r in sft_rows:
            f.write(json.dumps({"messages": r["messages"], "part": r.get("part"), "n_tokens": r.get("n_tokens")}, ensure_ascii = False) + "\n")
    json.dump(DATA, open(f"{CKPT_DIR}/sft_data.json", "w"), indent = 1, default = str)
    # 留出的题库（GRPO / DPO / 验证）也存一份；别的 notebook 会自己重新生成同样的文件（按题目哈希，结果相同）
    for name in ("rl_mcq.jsonl", "val_mcq.jsonl"):
        if os.path.exists(f"data/s1/{name}"):
            !cp data/s1/{name} "{CKPT_DIR}/{name}"
wandb.config.update({"data": DATA}, allow_val_change = True)
_tok = DATA.get("tokens") or 0
print(json.dumps(DATA, indent = 1, default = str))
print(f"训练集 {len(sft_rows)} 条，约 {_tok / 1e6:.1f}M token；按 6.2k token/秒估计训练约 {_tok / 6200 / 3600:.1f} 小时")

Generating train split:  18%|▏| 225000/1253230 [00:01<00:03, 263170.06 examples
Generating train split:  21%|▏| 266000/1253230 [00:01<00:03, 263369.15 examples
Generating train split:  24%|▏| 307000/1253230 [00:01<00:03, 262747.18 examples
Generating train split:  28%|▎| 349000/1253230 [00:01<00:03, 264793.93 examples
Generating train split:  30%|▎| 376000/1253230 [00:01<00:03, 262946.96 examples
Generating train split:  33%|▎| 408000/1253230 [00:01<00:03, 276062.45 examples
Generating train split:  36%|▎| 450000/1253230 [00:01<00:02, 272747.48 examples
Generating train split:  39%|▍| 491000/1253230 [00:01<00:02, 269382.23 examples
Generating train split:  43%|▍| 533000/1253230 [00:02<00:02, 259225.89 examples
Generating train split:  45%|▍| 565000/1253230 [00:02<00:02, 270674.43 examples
Generating train split:  48%|▍| 606000/1253230 [00:02<00:02, 269382.93 examples
Generating train split:  52%|▌| 648000/1253230 [00:02<00:02, 268965.83 examples
Generating train split:  54%|▌| 679000/1

JSONDecodeError: Unterminated string starting at: line 1 column 250 (char 249)

## 3. SFT

S1：LoRA r=128、学习率 1e-4（cosine）、有效 batch 32、1 个 epoch。think 变体：在合并后的 S1 上加一个新的 LoRA（r=64），学习率 5e-5，
每个 micro-batch 2 条（样本最长 8192 token）、有效 batch 32。只在回答部分算 loss（`train_on_responses_only`）。
每 `save_steps` 步存一个检查点；目录里已经有检查点时自动续训。

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only

class TimeLimit(TrainerCallback):
    # 到时间就让 Trainer 在当前步结束后停下（后面照常合并 / 推送 / 评测）
    def __init__(self, hours):
        self.t0, self.limit, self.hit = time.time(), hours * 3600, False
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.limit:
            self.hit = True
            control.should_training_stop = True

LORA_DIR = f"{CKPT_DIR}/s1_lora"
SFT_DONE = CACHE.has("sft") and os.path.exists(f"{LORA_DIR}/adapter_model.safetensors")
if SFT_DONE:
    SFT = CACHE.load("sft")
    print("[i] 训练已经完成过，直接读回 adapter（RESUME=False 可以强制重训）：", SFT, restore_lora(model, LORA_DIR))
_resume = RESUME and any(d.startswith("checkpoint-") for d in os.listdir(CKPT_DIR))
if not SFT_DONE and not _resume:
    print("fresh LoRA:", restore_lora(model))     # 重跑这一格时从头开始，而不是接着内存里上一次的权重

sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in sft_rows])
_limit = TimeLimit(MAX_TRAIN_HOURS)
trainer = None if SFT_DONE else SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_ds,
    callbacks = [_limit],
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = min(MAX_SEQ_LEN, Q["sft_max_tokens"] + 64),
        per_device_train_batch_size = Q["batch"],
        gradient_accumulation_steps = Q["accum"],
        warmup_steps = Q["warmup_steps"],
        max_steps = Q["max_steps"],
        num_train_epochs = 1,
        learning_rate = Q["lr"],
        lr_scheduler_type = "cosine",
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 3407,
        report_to = REPORT_TO,
        run_name = RUN_NAME,
        output_dir = CKPT_DIR,
        save_strategy = "steps",
        save_steps = Q["save_steps"],
        save_total_limit = 2,
    ),
)
if not SFT_DONE:
    instr_part, resp_part = RESPONSE_PARTS[CHAT_STYLE]
    trainer = train_on_responses_only(trainer, instruction_part = instr_part, response_part = resp_part)
    try:   # 训练标签的最后一个 token 应该是 eos
        _labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("label tail:", repr(tokenizer.decode(_labels[-12:])), "| ends with eos:", _labels[-1] == EOS_ID)
    except Exception as e:
        print("label check skipped:", e)

    print("resume from checkpoint:", _resume)
    sft_stats = trainer.train(resume_from_checkpoint = _resume)
    _losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    _total = trainer.state.max_steps
    SFT = dict(steps = trainer.state.global_step, total_steps = _total, finished = trainer.state.global_step >= _total,
               stopped_by_time_limit = _limit.hit, minutes_this_session = round(sft_stats.metrics["train_runtime"] / 60, 1),
               loss_first = _losses[0] if _losses else None,
               loss_last = round(statistics.fmean(_losses[-10:]), 4) if _losses else None, resumed = _resume)
    print(SFT)
    model.save_lora(LORA_DIR)                    # adapter 直接存在持久目录
    if SFT["finished"]:
        CACHE.save("sft", SFT)                   # 只有训完才算这个阶段完成；被时间上限打断的话下次从检查点续训
    else:
        print(f"[!] 训练在 {SFT['steps']}/{_total} 步停下（时间上限 {MAX_TRAIN_HOURS} 小时）。下面照常合并和推送这个中间检查点；"
              "想训完就把 MAX_TRAIN_HOURS 调大，重新 Run all（会从检查点续训）。")
    del trainer
del sft_ds
torch.cuda.empty_cache(); import gc; gc.collect()

## 4. 合并 → 推到 HF 私有仓库（下游 notebook 的起点）

`merged_16bit` 把 LoRA 合并进权重，tokenizer 带着模板和 eos=`<|endoftext|>`。推送失败（例如 token 没有 write 权限）时，合并模型存到检查点目录，
下游 notebook 的 `INIT_MODEL` 填这个路径也可以。

In [ ]:
HANDOFF = SFT_REPO
_push_key = f"pushed_{SFT.get('steps')}"
if CACHE.has(_push_key):
    HANDOFF = CACHE.load(_push_key)["handoff"]
    print("[i] 这个检查点已经推送 / 保存过：", HANDOFF)
else:
    try:
        try:
            model.push_to_hub_merged(SFT_REPO, tokenizer, save_method = "merged_16bit", private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数
            model.push_to_hub_merged(SFT_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
        print(f"pushed: https://huggingface.co/{SFT_REPO}")
    except Exception as e:
        HANDOFF = f"{CKPT_DIR}/merged_16bit"
        print("[!] 推送失败：", type(e).__name__, str(e)[:300])
        model.save_pretrained_merged(HANDOFF, tokenizer, save_method = "merged_16bit")
        print("合并模型存在", HANDOFF)
    CACHE.save(_push_key, {"handoff": HANDOFF})
print("=" * 80)
if THINK:
    print(f'思维链变体：下游 notebook 想从它开始，填 INIT_MODEL = "{HANDOFF}" 并把 GEN_BUDGET 设成 {Q["eval_max_tokens"]}（回答长，评测和 GRPO 都会慢很多）')
else:
    print(f'下游 notebook（11 GRPO / 12 DPO / 13 蒸馏）：INIT_MODEL = "auto" 会找 {SFT_REPO}；如果上面推送失败，改成 INIT_MODEL = "{HANDOFF}"')
print("=" * 80)

## 5. 评测和诊断（只输出聚合数字；不记录 GPQA 题目或模型输出）

- `stop_check`：teacher forcing 下，训练样本答案结尾预测 eos 的概率（应接近 1）。
- GPQA 快速评测：每题采样 `eval_samples` 次，`acc_flex` 是平均，`acc_flex_se` 是标准误。重点看**回答长度**（`mean_tokens`，S1 预期 600–1300）、
  截断率、以及 `acc_flex ÷ extract_rate`（有答案的题里的准确率；X1b 是 31%）。
- think 变体：和训练前的 S1 做配对比较；`acc_flex_at_8192` 是同样的回答按排行榜预算截断后的分数，两者之差就是"放宽预算值多少分"。
- `diagnose`：3 道 SciQ 验证题（公开数据）的回答结尾，看模型是不是在逐个分析选项。

In [ ]:
from datasets import load_dataset
from recipe_utils import sciq_mcq

lora_req = model.load_lora(LORA_DIR)
STOP = {}
try:
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    p_eos = []
    for r in sft_rows[:16]:
        ids = tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize = False), return_tensors = "pt",
                        add_special_tokens = False).input_ids[:, -2048:].to(model.device)
        if ids[0, -1].item() != EOS_ID:
            continue
        with torch.no_grad():
            p_eos.append(torch.softmax(model(input_ids = ids[:, :-1]).logits[0, -1].float(), -1)[EOS_ID].item())
    STOP = dict(n = len(p_eos), p_eos_mean = round(statistics.fmean(p_eos), 4) if p_eos else None,
                p_eos_median = round(statistics.median(p_eos), 4) if p_eos else None)
except Exception as e:
    print("stop check skipped:", type(e).__name__, e)
finally:
    os.environ.pop("UNSLOTH_RETURN_LOGITS", None)
print("stop check:", STOP)

COMPARE = {}
if RUN_EVAL:
    quick_eval("1_think" if THINK else "1_sft", lora_req)
    if THINK:
        print("思维链 vs S1（同一批题、同样的预算；配对差值，单位：分）：")
        COMPARE = EVAL.compare_all()
        for cfg, v in RESULTS["1_think"].items():
            print(cfg, "| 16384 预算:", v["acc_flex"], "| 按 8192 截断:", v.get("acc_flex_at_8192"), "| 超过 8192 的回答:",
                  v.get("over_8192_rate"), "% | 平均 token:", v["mean_tokens"])

val = load_dataset("allenai/sciq", split = "validation").select(range(3))
prompts = [chat_prompt(tokenizer, sciq_mcq(ex, 10_000 + i)[0]) for i, ex in enumerate(val)]
sp = SamplingParams(temperature = 0.0, max_tokens = 4096 if THINK else 1024, stop_token_ids = STOP_IDS, stop = STOP_STRS,
                    skip_special_tokens = False)
for o in model.fast_generate(prompts, sampling_params = sp, lora_request = lora_req):
    c = o.outputs[0]
    print(f"--- finish={c.finish_reason} tokens={len(c.token_ids)}")
    print(repr(c.text[-400:]))

## 6. 汇总（最后打印的 JSON 可以贴回给 Claude）

In [ ]:
rows = print_table(RESULTS) if RESULTS else []
REPORT = dict(notebook = "15_sft_checkpoint", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, base_model = BASE_MODEL,
              variant = VARIANT, init_model = INIT_MODEL, sft_repo = SFT_REPO, handoff = HANDOFF, chat_style = CHAT_STYLE, q = Q,
              data = DATA, sft = SFT, stop_check = STOP, quick_eval = RESULTS, compare = COMPARE, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS,
              wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    if rows:
        wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
json.dump(REPORT, open(f"{CKPT_DIR}/report_{EXP_NAME}.json", "w"), indent = 1, default = str)   # 断开之后也能在 Drive 里找到
print("=" * 30, "把下面整段 JSON 贴回给 Claude（也存在 Drive：", f"{CKPT_DIR}/report_{EXP_NAME}.json", "）", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 7. 断开 runtime

模型、adapter、报告都已经在 HF / Google Drive 上。`AUTO_DISCONNECT=True` 时这一格在 60 秒后释放 runtime，不再消耗 CU——
所以可以 **Run all 之后离开**：中途断线就重新打开、再 Run all，已经完成的阶段会直接跳过。

In [ ]:
end_runtime(AUTO_DISCONNECT)